In [1]:
import pandas as pd
import numpy as np

In [2]:
use_cols = [
    "item_id",
    "store_id",
    "date",
    "sales",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1"
]

df = pd.read_csv(
    "../data/interim/sales_long.csv",
    usecols=use_cols,
    parse_dates=["date"]
)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14944\3070002978.py:14: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df = df[df["store_id"] == "CA_1"]

In [4]:
def create_lag_features(df):
    df = df.sort_values("date")
    
    # Lag features
    for lag in [7, 14, 28]:
        df[f"lag_{lag}"] = df["sales"].shift(lag)
    
    # Rolling features
    for window in [7, 14, 28]:
        df[f"rmean_{window}"] = (
            df["sales"]
            .shift(1)
            .rolling(window)
            .mean()
        )
        
    return df


In [5]:
df = (
    df
    .groupby("item_id", group_keys=False)
    .apply(create_lag_features)
)



C:\Users\ASUS\AppData\Local\Temp\ipykernel_14944\2671925193.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(create_lag_features)


In [6]:
df["is_event"] = df["event_name_1"].notna().astype(int)


In [7]:
df = df.dropna().reset_index(drop=True)

In [8]:
df.shape


(460399, 17)

In [9]:
df.to_csv(
    "../data/processed/train_fe.csv",
    index=False
)

## Feature Engineering Summary

- Filtered the dataset to a single representative store (`CA_1`) to ensure
  temporal continuity and manageable memory usage
- Created lag features (7, 14, 28 days) at the item level to capture short- and
  medium-term temporal dependencies
- Created rolling mean features (7, 14, 28 days) using shifted sales values to
  smooth intermittent demand patterns
- Added calendar-based features (weekday, month, year) already present in the data
- Created a binary event indicator (`is_event`) based on the presence of calendar events
- Feature engineering was performed group-wise at the item level to preserve
  correct time-series ordering
- Rows with insufficient historical data (caused by lag and rolling windows)
  were removed to avoid data leakage

These engineered features form the core input for downstream forecasting
and baseline model evaluation.
